In [ ]:
from pathlib import Path
import h5py
import numpy as np

# Resolve the measurements folder relative to this notebook.
data_dir = (Path.cwd() / "../Data/measurements_h5").resolve()
print(f"Data folder: {data_dir}")

Data folder: /home/twalker/RbCaF/pytweezer/Data/measurements_h5


In [32]:
h5_files = sorted(data_dir.glob("*.h5"))
print(f"Found {len(h5_files)} .h5 files")
for i, f in enumerate(h5_files[:10], start=1):
    print(f"{i:2d}. {f.name}")

if not h5_files:
    raise FileNotFoundError(f"No .h5 files found in {data_dir}")

def get_h5_file_by_task(task_number, directory=None) -> h5py.File:
    directory = directory or (Path.cwd() / "../Data/measurements_h5").resolve()
    task_number = int(task_number)
    matches = sorted(Path(directory).glob(f"*task{task_number}*.h5"))

    if not matches:
        raise FileNotFoundError(
            f"No .h5 file found for task {task_number} in {Path(directory).resolve()}"
        )
    if len(matches) > 1:
        print(f"Warning: found {len(matches)} matches, using {matches[0].name}")

    return h5py.File(matches[0], "r")

Found 2 .h5 files
 1. FakeTweezerArray_task108.h5
 2. FakeTweezerArray_task109.h5


In [34]:
def get_h5_dataset_by_task(task_number, dataset_path, directory=None):
    with get_h5_file_by_task(task_number, directory) as h5_file:
        if dataset_path not in h5_file:
            raise KeyError(f"Dataset '{dataset_path}' not found in {h5_file.filename}")
        return h5_file[dataset_path][()]

In [ ]:
from datetime import datetime

def _as_text(value):
    if isinstance(value, (bytes, np.bytes_)):
        return value.decode("utf-8", errors="replace")
    return str(value)

def _read_attr(group, key, default="n/a"):
    if key not in group.attrs:
        return default
    return _as_text(group.attrs[key])

def print_h5_summary(task_number, directory=None):
    with get_h5_file_by_task(task_number, directory) as h5f:
        rep_names = sorted([k for k in h5f.keys() if k.startswith("rep_")])
        run_count = sum(
            len([rk for rk in h5f[rep].keys() if rk.startswith("run_")])
            for rep in rep_names
        )

        print(f"File: {h5f.filename}")
        task_info = h5f.get('task_info')

        if task_info is not None:
            print(f"Experiment: {_read_attr(task_info, 'experiment_name')}")
        print(f"Task: {_read_attr(h5f, 'task')}")
        print(f"Total reps: {len(rep_names)}")
        print(f"Total runs: {run_count}")

        if not rep_names:
            return

        first_rep = rep_names[0]
        first_run = sorted([k for k in h5f[first_rep].keys() if k.startswith("run_")])[0]
        start_group = h5f[f"{first_rep}/{first_run}/start"]
        end_group = h5f[f"{first_rep}/{first_run}/end"]

        start_ts = start_group.attrs.get("_starttime")
        end_ts = end_group.attrs.get("_endtime")

        if start_ts is not None:
            print("Start time:", datetime.fromtimestamp(float(start_ts)).isoformat(timespec="seconds"))
        if end_ts is not None:
            print("End time:  ", datetime.fromtimestamp(float(end_ts)).isoformat(timespec="seconds"))
        if start_ts is not None and end_ts is not None:
            print(f"Duration: {float(end_ts) - float(start_ts):.3f} s")

        if task_info is not None:
            git_group = task_info.get('git')
            if git_group is not None:
                print(f"Git hash: {_read_attr(git_group, 'commit_short', _read_attr(git_group, 'commit'))}")
                print(f"Branch: {_read_attr(git_group, 'branch')}")
            
            if "arguments" in task_info:
                arg_keys = sorted(task_info["arguments"].attrs.keys())
                print(f"Scan args ({len(arg_keys)}): {arg_keys}")
            if "mm_params" in task_info:
                mm_keys = sorted(task_info["mm_params"].attrs.keys())
                print(f"MM params ({len(mm_keys)}): {mm_keys}")

# Example
print_h5_summary(task_number=109)

File: /home/twalker/RbCaF/pytweezer/Data/measurements_h5/FakeTweezerArray_task109.h5
Experiment: FakeTweezerArray
Task: 109
Total reps: 0
Total runs: 0
